# Task 01 — Ticket Classification Prompt Design

Design and test a production-ready ticket classifier using the OpenAI API.

**You will implement**:
1. `system_prompt` — system instructions for the classifier
2. `user_template` — per-ticket user message template with `{ticket}` placeholder
3. `classify_ticket(client, text)` → `dict` — real API call, returns parsed JSON
4. Accuracy test on all 20 labeled tickets (≥ 70% category accuracy required)
5. Chain-of-thought variant with reasoning field

## Setup

In [ ]:
from openai import OpenAI
import json, os

# SET YOUR API KEY HERE
api_key = "your-api-key-here"
client = OpenAI(api_key=api_key)

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

with open(os.path.join(FIXTURES, "tickets.json")) as f:
    tickets = json.load(f)

def parse_json_safe(text: str) -> dict | None:
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

print(f"✓ Setup complete. {len(tickets)} tickets loaded.")

## Task 1.1 — Design the System Prompt

Write `system_prompt` that:
- Defines the classifier's role clearly
- Lists all 4 categories with brief descriptions
- Lists all 3 priorities with descriptions
- Specifies the exact JSON output: `{"category": "...", "priority": "..."}`
- Is at least 100 characters

In [ ]:
# YOUR CODE HERE
system_prompt = "..."

# TEST — Do not modify
assert isinstance(system_prompt, str)
assert len(system_prompt.strip()) >= 100, f"system_prompt too short ({len(system_prompt)} chars, need >= 100)"

for kw in ["json", "category", "priority"]:
    assert kw in system_prompt.lower(), f"system_prompt must mention '{kw}'"
for cat in ["billing", "technical", "account", "shipping"]:
    assert cat in system_prompt.lower(), f"system_prompt must list category '{cat}'"
for pri in ["high", "medium", "low"]:
    assert pri in system_prompt.lower(), f"system_prompt must list priority '{pri}'"

print("✓ Task 1.1 passed")

## Task 1.2 — Design the User Template

Write `user_template` with a `{ticket}` placeholder that frames the ticket for the classifier.

In [ ]:
# YOUR CODE HERE
user_template = "..."

# TEST — Do not modify
assert isinstance(user_template, str)
assert "{ticket}" in user_template, "user_template must contain {ticket} placeholder"
formatted = user_template.format(ticket="test content")
assert "test content" in formatted
assert len(formatted) > len("test content"), "Template must add context around the ticket text"

print("✓ Task 1.2 passed")

## Task 1.3 — Implement classify_ticket()

```python
def classify_ticket(client, ticket_text: str) -> dict:
```

- Use `system_prompt` and `user_template`
- Call `client.chat.completions.create()` with `temperature=0.0`
- Parse the JSON response and return a dict

In [ ]:
# YOUR CODE HERE
def classify_ticket(client, ticket_text: str) -> dict:
    ...

# TEST — real API call
result = classify_ticket(client, tickets[0]["text"])

assert isinstance(result, dict), f"classify_ticket must return dict, got {type(result)}"
assert "category" in result, f"Result must have 'category' key. Got: {result}"
assert "priority" in result, f"Result must have 'priority' key. Got: {result}"
assert result["category"] in {"billing", "technical", "account", "shipping"},     f"Invalid category: {result['category']!r}"
assert result["priority"] in {"high", "medium", "low"},     f"Invalid priority: {result['priority']!r}"

print(f"✓ Task 1.3 passed")
print(f"  Ticket:   {tickets[0]['text'][:65]}...")
print(f"  Got:      category={result['category']}, priority={result['priority']}")
print(f"  Expected: category={tickets[0]['category']}, priority={tickets[0]['priority']}")

## Task 1.4 — Accuracy Test on All 20 Tickets

Classify all 20 labeled tickets. Category accuracy must be **≥ 70%**.

In [ ]:
# TEST — real API calls on all 20 tickets
cat_correct = 0
pri_correct = 0
errors = []

for t in tickets:
    result = classify_ticket(client, t["text"])
    if result.get("category") == t["category"]:
        cat_correct += 1
    else:
        errors.append({"ticket": t["text"][:50], "expected": t["category"], "got": result.get("category")})
    if result.get("priority") == t["priority"]:
        pri_correct += 1

cat_acc = cat_correct / len(tickets)
pri_acc = pri_correct / len(tickets)

print(f"Category accuracy: {cat_correct}/{len(tickets)} = {cat_acc:.0%}")
print(f"Priority accuracy: {pri_correct}/{len(tickets)} = {pri_acc:.0%}")

if errors:
    print(f"\nCategory errors ({len(errors)}):")
    for e in errors:
        print(f"  expected={e['expected']}, got={e['got']}: {e['ticket']}")

assert cat_acc >= 0.70, f"Category accuracy {cat_acc:.0%} < 70% — improve your system_prompt"
print("\n✓ Task 1.4 passed")

## Task 1.5 — Chain-of-Thought Variant

Write `cot_system_prompt` that asks the model to reason step-by-step.
Output must include a `"reasoning"` field plus `"category"` and `"priority"`.

Implement `classify_with_cot(client, ticket_text)` using this prompt.

In [ ]:
# YOUR CODE HERE
cot_system_prompt = "..."

def classify_with_cot(client, ticket_text: str) -> dict:
    ...

# TEST — real API call
result = classify_with_cot(client, tickets[5]["text"])

assert isinstance(result, dict), f"classify_with_cot must return dict, got {type(result)}"
assert "reasoning" in result, f"CoT result must have 'reasoning' key. Got keys: {list(result.keys())}"
assert "category" in result, f"CoT result must have 'category' key. Got keys: {list(result.keys())}"
assert "priority" in result, f"CoT result must have 'priority' key. Got keys: {list(result.keys())}"
assert len(result["reasoning"]) >= 20, "reasoning field is too short — model must explain its thinking"
assert result["category"] in {"billing", "technical", "account", "shipping"}

print(f"✓ Task 1.5 passed")
print(f"  Reasoning: {result['reasoning'][:120]}...")
print(f"  Category:  {result['category']}")
print(f"  Priority:  {result['priority']}")

## Done

In [ ]:
print("\n✓ All task_01 tests passed!")